In [1]:
!pip -q install -U torch transformers accelerate scikit-learn pandas numpy



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import json, random, shutil
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

PROYECTO_DIR = Path.cwd()
if not (PROYECTO_DIR / "train.csv").exists():
    PROYECTO_DIR = PROYECTO_DIR / "Proyecto"

MODEL_NAME = "distilbert/distilbert-base-multilingual-cased"
MAX_LEN = 256
AUG_FRACTION = 0.35


c:\ANDES\BI\MaterialDeClase-ISIS-2611\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_df = pd.read_csv(PROYECTO_DIR / "train.csv")
eval_df = pd.read_csv(PROYECTO_DIR / "eval.csv")

train_df["text"] = train_df["text"].fillna("").astype(str)
eval_df["text"] = eval_df["text"].fillna("").astype(str)

classes = [int(x) for x in sorted(train_df["decade"].unique())]
label2id = {c: i for i, c in enumerate(classes)}
id2label = {i: c for c, i in label2id.items()}

train_df["label"] = train_df["decade"].map(label2id)

print("train:", train_df.shape)
print("eval:", eval_df.shape)
print("clases:", len(classes), classes[0], classes[-1])
print(train_df["decade"].value_counts().sort_index().describe())


train: (31403, 3)
eval: (3490, 2)
clases: 39 150 188
count     39.000000
mean     805.205128
std       22.058749
min      754.000000
25%      787.000000
50%      807.000000
75%      823.500000
max      848.000000
Name: count, dtype: float64


In [4]:
def augment_text(text, rng):
    pares = {
        "s": "f", "f": "s",
        "u": "v", "v": "u",
        "i": "y", "y": "i",
        "c": "ç"
    }
    
    chars = []
    for ch in str(text):
        if ch.isalpha() and rng.random() < 0.012:
            continue
        
        low = ch.lower()
        if low in pares and rng.random() < 0.025:
            rep = pares[low]
            ch = rep.upper() if ch.isupper() else rep
        chars.append(ch)
    
    words = "".join(chars).split()
    for i in range(len(words) - 1):
        if rng.random() < 0.008:
            words[i], words[i + 1] = words[i + 1], words[i]
    return " ".join(words)

train_base, val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["label"],
    random_state=SEED
)

rng = random.Random(SEED)
aug_df = train_base.sample(frac=AUG_FRACTION, random_state=SEED).copy()
aug_df["text"] = aug_df["text"].apply(lambda x: augment_text(x, rng))

train_aug = pd.concat([train_base, aug_df], ignore_index=True)
train_aug = train_aug.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("train original:", train_base.shape)
print("train aumentado:", train_aug.shape)
print("validacion:", val_df.shape)

train original: (26692, 3)
train aumentado: (36034, 3)
validacion: (4711, 3)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.enc = tokenizer(
            list(texts),
            truncation=True,
            padding=True,
            max_length=MAX_LEN
        )
        self.labels = None if labels is None else list(labels)
    
    def __len__(self):
        return len(self.enc["input_ids"])
    
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    
train_ds = TextDataset(train_aug["text"], train_aug["label"])
val_ds = TextDataset(val_df["text"], val_df["label"])

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(classes),
    id2label={i: str(id2label[i]) for i in id2label},
    label2id={str(k): v for k, v in label2id.items()}
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

args = TrainingArguments(
    output_dir=str(PROYECTO_DIR / "runs_parte2_distilmbert_aug"),
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    weight_decay=0.01,
    warmup_ratio=0.08,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    optim="adamw_torch"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.evaluate()

In [ ]:
val_pred = trainer.predict(val_ds)
val_pred_labels = np.argmax(val_pred.predictions, axis=1)

print(classification_report(
    val_df["label"],
    val_pred_labels,
    target_names=[str(c) for c in classes],
    zero_division=0
))

In [ ]:
eval_ds = TextDataset(eval_df["text"])

pred = trainer.predict(eval_ds)
pred_labels = np.argmax(pred.predictions, axis=1)
answers = [id2label[int(i)] for i in pred_labels]

submission = pd.DataFrame({
    "id": eval_df["id"],
    "answer": answers
})

submission_path = PROYECTO_DIR / "submission_parte2_distilmbert_aug.csv"
submission.to_csv(submission_path, index=False)

submission.head()

In [ ]:
save_dir = PROYECTO_DIR / "modelo_parte2_distilmbert_aug"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

with open(save_dir / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "classes": classes,
            "label2id": {str(k): int(v) for k, v in label2id.items()},
            "id2label": {str(k): int(v) for k, v in id2label.items()}
        },
        f,
        ensure_ascii=False,
        indent=2
    )

zip_path = shutil.make_archive(str(save_dir), "zip", save_dir)

print("Modelo guardado en:", save_dir)
print("ZIP del modelo:", zip_path)
print("Submission:", submission_path)